### 13.2.5. `Evaluator-optimizer`：评估器—优化器
就是循环，
#### 13.2.5.1. 核心思想

一个节点生成结果，另一个节点评估结果；不合格就携带反馈重新生成：

```text
          ┌──────────────────────┐
          ↓                      │
输入 → Generator → Evaluator ────┤
                      ├─ 不合格 ─┘
                      └─ 合格 → END
```

适合：

* 翻译质量迭代
* 代码生成与代码审查
* 文案生成与合规检查
* **`SQL`** 生成与语法验证
* 报告生成与事实检查
* 人工审批后修改

这种模式适用于存在明确质量标准，但通常需要多轮修改才能满足标准的任务。评估者既可以是 **`LLM`**，也可以是规则、测试程序或人类。

#### 13.2.5.2. 必要的控制

为了避免无限循环，必须引入递归限制，可以是

- 优雅退出的主动方法
- 异常中断的被动方法

我们在介绍循环结构时已经系统讲解过了。

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langchain_deepseek import ChatDeepSeek

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from loguru import logger
load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

# 图状态
class OverAllState(TypedDict):
    joke: str
    topic: str
    feedback: str
    funny_or_not: str

# 定义用于结构化输出的 Schema，作为笑话评估依据
class Feedback(BaseModel):
    grade: Literal["好笑", "不好笑"] = Field(
        description="判断这个笑话是否好笑。",
    )
    feedback: str = Field(
        description="如果笑话不好笑，请给出具体的改进建议。",
    )

# 为大模型添加结构化输出能力，定义评估器
evaluator = model.with_structured_output(Feedback) # 结构化输出

# 节点
def model_call_generator(state: OverAllState) -> OverAllState:
    """大模型生成笑话"""

    if state.get("feedback"):
        msg = model.invoke(
            f"""
            写一个关于“{state['topic']}”的笑话。

            请参考下面的改进建议：
            {state['feedback']}
            """
        )
    else:
        msg = model.invoke(f"写一个关于“{state['topic']}”的笑话")

    return {"joke": msg.content}


def model_call_evaluator(state: OverAllState) -> OverAllState:
    """大模型评估笑话"""

    grade = evaluator.invoke(
        f"""
        请评估下面这个笑话是否好笑：

        {state['joke']}
        """
    )
    logger.info(grade)
    return {
        # "funny_or_not":grade.grade,
        "funny_or_not":"不好笑", # 测试使用， 验证被动退出
        "feedback": grade.feedback,
    }


# 条件边函数：根据评估结果决定结束流程，或者返回笑话生成节点
def route_joke(state: OverAllState) -> Literal["accept", "reject_and_feedback", END]:
    """根据评估结果决定接受笑话或根据反馈重新生成"""

    if state["funny_or_not"] == "好笑":
        return "accept"
    elif state["funny_or_not"] == "不好笑":
        return "reject_and_feedback"  # 优雅主动退出
    return END

# 构建工作流
builder = StateGraph(state_schema=OverAllState)

# 添加节点
builder.add_node("model_call_generator", model_call_generator)
builder.add_node("model_call_evaluator", model_call_evaluator)

# 添加边，连接各个节点
builder.add_edge(START, "model_call_generator")
builder.add_edge("model_call_generator", "model_call_evaluator")
builder.add_conditional_edges(
    "model_call_evaluator",
    route_joke,
    {
        # route_joke 返回的名称：接下来要执行的节点
        "accept": END,
        "reject_and_feedback": "model_call_generator",
    },
)

# 编译工作流
graph = builder.compile()

# 调用工作流
state = None
try:
    state = graph.invoke({"topic": "猫"},config={"recursion_limit":10})
except Exception as e:
    logger.info("超步数量达到最大限制")  # 被动退出

print(state["joke"])

# 显示工作流图
from IPython.display import display
display(graph)

2026-09-20 09:58:40.399 | INFO     | __main__:model_call_evaluator:67 - grade='不好笑' feedback='这个笑话的核心问题在于“梗”的落点太软、太绕，缺少一个意料之外又符合逻辑的爆点。改进建议：1）把“等一个人”的悬念压缩，别铺三天那么长，前两天的重复（喝奶、付钱、走人）对笑点没有贡献，只会拖节奏；2）最后的反转“对面那家真不让进”是个“反高潮”，虽然有点冷幽默，但和前面“要亲眼看看他被打脸”的气势完全断裂，读者会觉得被耍而不是被逗；3）可以改成一个更干脆的反转，比如猫其实是在等那个酒保自己说漏嘴“猫不能进酒吧”，然后立刻掏出手机拍视频/叫来记者，把它变成一个“钓鱼执法”式的反转；4）语言上可以更简练，去掉冗余叙述，让节奏更快。总体是“设定有趣、执行松散”。'
2026-09-20 09:58:43.159 | INFO     | __main__:model_call_evaluator:67 - grade='好笑' feedback='这个笑话结构完整、节奏好：猫的行为反常（每天准时来点牛奶）制造悬念，酒保的疑问引出转折，而“猫掏出手机录证据”是出乎意料又合理的荒诞设定。最后的双重反转——先是“你不是记者”，再是“我是自媒体”——既呼应了当下的自媒体/碰瓷式维权现象，又用猫舔爪子的动作保持角色一致性，收尾干净利落。若要微调，可把“XX酒吧”换成具体又离谱的店名（如“三花猫克星酒馆”），增强画面感，但整体已经很好笑。'
2026-09-20 09:58:46.640 | INFO     | __main__:model_call_evaluator:67 - grade='好笑' feedback='这个笑话结构完整，建立了"悬念—反转—再反转"的三段式：猫收集证据→证明24小时营业→最后揭示是自媒体碰瓷，层层递进。笑点在于把网络时代"自媒体敲诈/碰瓷"的社会现象，套用到一只猫身上，荒诞与现实产生反差。凌晨三点的换行、牛奶、猫爪舔舐等细节也增强了画面感。结尾"不是，我是自媒体"是点睛之笔，反转干净利落。若想更好笑，可以让酒保最后还有一句崩溃台词（如"那你到底想怎样"），或加一句猫开价的收尾，让敲诈逻辑闭环。但整体已经达到好笑标准。'
2026-09-20 09:58:49.8

TypeError: 'NoneType' object is not subscriptable